In [0]:
# endpoint_utils
import requests
import re
from pyspark.sql.functions import udf, col, expr
from pyspark.sql.types import ArrayType, StringType

def fetch_wiki_with_fallback(page_name, regex_pattern):
    if not page_name: return []
    
    # IMPORTANTE: Definir User-Agent para evitar bloqueio da Wiki
    headers = {"User-Agent": "DatabricksDataPipeline/1.0 (contact: your@email.com)"}
    session = requests.Session()
    session.headers.update(headers)
    """
    Tenta extrair o link da página exata. 
    Se falhar, busca as 3 páginas mais relevantes na Wiki e tenta extrair delas.
    """
    url = "https://en.wikipedia.org/w/api.php"
    found_links = []

    def get_links_from_page(target_page):
        # Tenta extrair tanto de 'text' (corpo) quanto de 'externallinks' (metadados)
        params = {
            "action": "parse", "page": target_page, "format": "json", 
            "prop": "text|externallinks", "redirects": 1
        }
        try:
            res = session.get(url, params=params, timeout=10).json()
            if "parse" not in res: return []
            
            # 1. Busca no HTML (Regex)
            html = res["parse"]["text"]["*"]
            links = re.findall(regex_pattern, html)
            
            # 2. Busca nos Links Externos oficiais da Wiki
            ex_links = res["parse"].get("externallinks", [])
            for link in ex_links:
                match = re.search(regex_pattern, link)
                if match: links.append(match.group(1))
            
            return links
        except: return []

    # Passo 1: Tentativa na página exata
    found_links.extend(get_links_from_page(page_name))

    # Passo 2: Fallback - Se não achou nada, pesquisa na Wiki por termos similares
    if not found_links:
        search_params = {
            "action": "query", "list": "search", "format": "json",
            "srsearch": page_name.replace("_", " "), "srlimit": 3
        }
        try:
            search_res = session.get(url, params=search_params, timeout=10).json()
            candidates = [r["title"] for r in search_res.get("query", {}).get("search", [])]
            
            for candidate in candidates:
                found_links.extend(get_links_from_page(candidate))
                if found_links: break # Para assim que encontrar o primeiro
        except: pass

    # Limpa duplicatas e lixo (como archive.org)
    unique_links = list(set([l for l in found_links if "archive" not in l.lower()]))
    return unique_links

# Registro da UDF
fetch_social_udf = udf(fetch_wiki_with_fallback, ArrayType(StringType()))

def run_enhanced_extraction(source_table, target_table, regex, output_col):
    # 1. Coleta candidatos (Execução Distribuída)
    df_raw = spark.table(source_table).select("wiki_page").distinct() \
                  .withColumn("candidates", fetch_social_udf(col("wiki_page"), expr(f"'{regex}'"))) \
                  .withColumn("exploded", expr("explode_outer(candidates)"))

    # 2. SQL para Similaridade (Levenshtein) - Garante o melhor match
    df_raw.createOrReplaceTempView("v_social_candidates")
    
    query = f"""
        WITH ranked AS (
            SELECT 
                wiki_page, 
                exploded as {output_col},
                levenshtein(lower(wiki_page), lower(exploded)) as score,
                row_number() OVER (PARTITION BY wiki_page ORDER BY levenshtein(lower(wiki_page), lower(exploded)) ASC) as rnk
            FROM v_social_candidates
            WHERE exploded IS NOT NULL
        )
        SELECT wiki_page, {output_col} FROM ranked WHERE rnk = 1
    """
    
    df_final = spark.sql(query)
    
    # 3. Overwrite Total
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    df_final.write.format("delta").saveAsTable(target_table)
    
    print(f"✅ Extração concluída com sucesso em {target_table}")